In [ ]:
# ============================================================================
# Cell 1: 导入库
# ============================================================================
from vnpy.trader.setting import SETTINGS
from vnpy.trader.constant import Exchange, Interval
from vnpy.trader.object import  HistoryRequest
from vnpy.trader.datafeed import get_datafeed
from vnpy.alpha.lab import AlphaLab
from vnpy.trader.database import DB_TZ
from vnpy.alpha import  logger
from datetime import datetime
import rqdatac as rq
from pathlib import Path
from tqdm import tqdm
import json

In [ ]:
# ============================================================================
# Cell 2: 配置
# ============================================================================


# 初始化数据服务
datafeed = get_datafeed()           #RqdataDatafeed类  C:\veighna_studio\Lib\site-packages\vnpy_rqdata\rqdata_datafeed.py
print(f'数据服务类型: {datafeed.__class__.__name__}')

# 尝试初始化
inited = datafeed.init(output=print)
print(f'初始化结果: {inited}')

In [ ]:
# ============================================================================
# Cell 3: 路径配置
# ===========================================================================

BASE_PATH = Path('D:/Aquant project/MF')
LAB_PATH = BASE_PATH / 'MF_lab'
MINUTE_PATH = LAB_PATH / 'minute'
DAILY_PATH = LAB_PATH / 'daily'

# 获取MF_Lab
lab = AlphaLab(str(LAB_PATH))

In [ ]:
# ============================================================================
# Cell 4: 数据下载
# ============================================================================

vt_index_symbol = "000300.SSE"
rq_index_symbol = "000300.XSHG"

# 总时间跨度
start = datetime(2018, 1, 1)
end = datetime.now()
interval1 = Interval.MINUTE                  #数据频率

# 回测跨度 回测需要日线数据算收益
test_start = datetime(2025, 1, 1)
test_end = datetime(2026, 4, 7)
interval2 = Interval.DAILY


# 4.1 下载成分股列表

# 下载总时间跨度的动态沪深300股票池股票代码
data = rq.index_components(rq_index_symbol, start_date=start, end_date=end)

# 将rq的合约代码转化内vnpy的合约代码
vt_index_components = {}
for dt, rq_symbols in data.items():
    vt_symbols: list = []

    for rq_symbol in rq_symbols:
        vt_symbol = rq_symbol.replace("XSHG", "SSE").replace("XSHE", "SZSE")
        vt_symbols.append(vt_symbol)

    vt_index_components[dt.strftime("%Y-%m-%d")] = vt_symbols    #index_components = {"%Y-%m-%d":[成分股列表]，....}


# 保存
lab.save_component_data(vt_index_symbol, vt_index_components)

# 加载成分股代码
component_symbols = lab.load_component_symbols(vt_index_symbol, start, end)
print(component_symbols)

In [ ]:
start = datetime(2018, 1, 1)
end = datetime.now()
trading_days = rq.get_trading_dates(  # 获取交易日
            start,
            end,
            market="cn"  # 'cn' 代表中国证券市场
        )
trading_days_str = [d.isoformat() for d in trading_days]
with open(lab.trading_days_path, "w+") as f:
    json.dump(trading_days_str, f, indent=2)

In [ ]:
print(lab.load_trading_days())

In [ ]:
# ============================================================================
# Cell 5: 回测参数配置
# ============================================================================
for vt_symbol in component_symbols:
    lab.add_contract_setting(
        vt_symbol,
        long_rate=5/10000,
        short_rate=15/10000,
        size=1,
        pricetick=0.0001,
    )

In [ ]:
# ============================================================================
# Cell 6:
# ============================================================================

# 4.2 下载k线
start = start.replace(tzinfo=DB_TZ)    # 下载要标明时区
end = end.replace(tzinfo=DB_TZ)

task_symbols = component_symbols    # 目标股票

n = 0

for vt_symbol in tqdm(task_symbols):
    symbol, exchange_str = vt_symbol.split(".")
    print(f'下载 {vt_symbol} ...')
    req = HistoryRequest(symbol, Exchange(exchange_str), start, end, interval1, adjust_type='post')
    bars = datafeed.query_bar_history(req)

    if bars:
        n += 1
        lab.save_bar_data(bars)
        print(f'股票{n}')
        print(f'  ->  {len(bars)} 条K线')
    else:
        logger.error(f"下载{vt_symbol}数据失败")

print(f'剩余{len(task_symbols)-n}只未下载')

In [ ]:
# ============================================================================
# Cell 7:
# ============================================================================

# 4.4 下载剩余k线

# 获取所有 .parquet 文件的名称部分（不含扩展名）
done_symbols = [f.stem for f in Path(MINUTE_PATH).glob("*.parquet")]
task_symbols = [vt_symbol for vt_symbol in component_symbols if vt_symbol not in done_symbols]

start = start.replace(tzinfo=DB_TZ)    # 下载要标明时区
end = end.replace(tzinfo=DB_TZ)

n = 0

for vt_symbol in tqdm(task_symbols):
    symbol, exchange_str = vt_symbol.split(".")
    print(f'下载 {vt_symbol} ...')
    req = HistoryRequest(symbol, Exchange(exchange_str), start, end, interval1)
    bars = datafeed.query_bar_history(req)
    if bars:
        n += 1
        lab.save_bar_data(bars)
        print(f'股票{n}')
        print(f'  ->  {len(bars)} 条K线')
    else:
        logger.error(f"下载{vt_symbol}数据失败")

print(f'剩余{len(task_symbols)-n}只未下载')

In [ ]:
# ============================================================================
# Cell 8:
# ============================================================================
# 4.3 下载回测所需日线
test_start = test_start.replace(tzinfo=DB_TZ)
test_end = test_end.replace(tzinfo=DB_TZ)

task_symbols = component_symbols + [vt_index_symbol]

n = 0

for vt_symbol in tqdm(task_symbols):
    symbol, exchange_str = vt_symbol.split(".")
    print(f'下载 {vt_symbol} ...')
    req = HistoryRequest(symbol, Exchange(exchange_str), test_start, test_end, Interval.DAILY)
    bars = datafeed.query_bar_history(req)

    if bars:
        n += 1
        lab.save_bar_data(bars)
        print(f'股票{n}')
        print(f'  ->  {len(bars)} 条K线')
    else:
        logger.error(f"下载{vt_symbol}数据失败")

print(f'剩余{len(task_symbols)-n}只未下载')

In [ ]:
quota = rq.user.get_quota()
print(quota)